<a href="https://colab.research.google.com/github/Kaitokidbua/ASEAN_Transport/blob/main/ASEAN_Part4_KualaLumpur_Fig15-18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

✅ Setup complete — Dark theme loaded


In [10]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

Dataset: 2,027 rows | 5 cities
Cities: ['Bangkok', 'Jakarta', 'Kuala Lumpur', 'Singapore', 'Manila']


In [11]:
# ── KL: prep data ─────────────────────────────────────────────────────────────
kl = df[(df['City']=='Kuala Lumpur') & (df['Year'].between(2019,2024))].copy()
kl['Mode_Label'] = kl['Mode'].map(lambda x: MODE_KL.get(x,x))

kl_m  = kl.groupby('Date')['Ridership'].sum().reset_index()
kl_yr = kl.groupby(['Year','Mode_Label'])['Ridership'].sum().reset_index()
top5_kl = kl.groupby('Mode_Label')['Ridership'].sum().nlargest(5).index.tolist()
kl_top  = kl_yr[kl_yr['Mode_Label'].isin(top5_kl)]
kl_sh   = kl.groupby('Mode_Label')['Ridership'].sum().reset_index().sort_values('Ridership',ascending=False)

yoy_list=[]
for mode in top5_kl:
    sub = kl_yr[kl_yr['Mode_Label']==mode].sort_values('Year').copy()
    sub['YoY'] = sub['Ridership'].pct_change()*100
    yoy_list.append(sub)
kl_yoy = pd.concat(yoy_list).dropna(subset=['YoY'])
print('Kuala Lumpur data ready ✅')

Kuala Lumpur data ready ✅


In [12]:
KL_CLR = ['#4ECDC4','#74B9FF','#FFB347','#C77DFF','#A8E6CF']

# 3.1 Monthly
fig = px.area(kl_m, x='Date', y='Ridership',
    title='<b>Fig.15</b>  กัวลาลัมเปอร์ — ผู้โดยสารรายเดือน (2019–2024)',
    labels={'Ridership':'จำนวนผู้โดยสาร','Date':''},
    color_discrete_sequence=[CITY_COLORS['Kuala Lumpur']])
fig.update_traces(fillcolor='rgba(78,205,196,0.13)', line_color=CITY_COLORS['Kuala Lumpur'], line_width=2.2)
fig.update_layout(**base_layout(420), hovermode='x unified',
                  yaxis=dict(tickformat='.2s',**ax_style()), xaxis=ax_style())
fig.add_vrect(x0='2020-01-01', x1='2021-06-01', fillcolor='#FF4D4D', opacity=0.07,
              annotation_text='COVID-19', annotation_position='top left',
              annotation_font_color='#FF8080', annotation_font_size=10)
fig.show()



In [13]:
fig = px.bar(
    kl_sh.head(6),
    x='Ridership',
    y='Mode_Label',
    orientation='h', # Horizontal bar chart
    title='<b>Fig.16</b>  กัวลาลัมเปอร์ — สัดส่วนผู้โดยสารแต่ละระบบ (2019–2024)',
    labels={'Ridership':'จำนวนผู้โดยสาร','Mode_Label':'ระบบขนส่ง'},
    color='Mode_Label', # Color by mode label
    color_discrete_sequence=['#4ECDC4','#74B9FF','#FFB347','#C77DFF','#A8E6CF'],
)

# Get the base layout configuration
layout_config = base_layout(460, legend_h=False)
# Update the legend properties
layout_config['legend'].update(orientation='v', y=0.5, yanchor='middle', x=1.02, xanchor='left') # Adjust legend for horizontal bar chart

fig.update_layout(
    **layout_config,
    yaxis={'categoryorder': 'total ascending', 'categoryarray': kl_sh.head(6)['Ridership'].tolist()},
    xaxis=dict(tickformat='.2s', **ax_style()),
    bargap=0.1 # Adjust gap between bars
)
fig.show()

In [28]:
fig = px.bar(
    kl_yr, x='Year', y='Ridership', color='Mode_Label',
    barmode='group', # Set to grouped bar chart as requested
    title='<b>Fig.17</b>  กัวลาลัมเปอร์ — ผู้โดยสารแต่ละระบบขนส่ง รายปี (2019–2024)',
    labels={'Ridership':'ผู้โดยสาร','Mode_Label':'ระบบขนส่ง','Year':''},
    color_discrete_sequence=KL_CLR, # Use Kuala Lumpur specific colors
)

# Get the base layout dictionary
layout_config = base_layout(490, legend_h=True)
# Update the top margin within the existing margin dictionary
layout_config['margin']['t'] = 120

fig.update_layout(**layout_config,
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()),
                  bargap=0.20)
fig.show()

In [15]:
# ── Kuala Lumpur: prep data for YoY (already done in a prior cell, using kl_yoy) ──
# top_modes_kl was used to create kl_yoy in cell rJenKLkNnKap
# yoy_list_kl was already created as kl_yoy

print('Kuala Lumpur YoY data ready ✅')

# 3.4 YoY Area Chart for Kuala Lumpur
fig = px.area(
    kl_yoy, x='Year', y='YoY', color='Mode_Label',
    title='<b>Fig.18</b>  กัวลาลัมเปอร์ — อัตราการเปลี่ยนแปลงผู้โดยสาร YoY (%) แต่ละระบบ',
    labels={'YoY':'YoY Change (%)','Mode_Label':'ระบบขนส่ง','Year':''},
    color_discrete_sequence=KL_CLR,
)
fig.add_hline(y=0, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(460),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix='%',**ax_style()))
fig.show()

Kuala Lumpur YoY data ready ✅
